In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# ─────────────────────────────────────────────────────────────────────
# STEP 1 — Load your hourly data
# Save your 8184 values to a CSV file called Load_data.csv (one value
# per row, no header), then run this script.
# ─────────────────────────────────────────────────────────────────────
df_raw = pd.read_csv(r"C:\Users\wb590499\Documents\Projects\EPM\epm\input\data_zim\Load_data.csv", header=None, names=["load"])
raw_values = df_raw["load"].values

# ─────────────────────────────────────────────────────────────────────
# STEP 2 — Reshape into daily profiles (341 days x 24 hours)
# ─────────────────────────────────────────────────────────────────────
n_hours = len(raw_values)
n_days  = n_hours // 24
assert n_days * 24 == n_hours, "Data length must be divisible by 24"

daily_profiles = raw_values[:n_days * 24].reshape(n_days, 24)

# ─────────────────────────────────────────────────────────────────────
# STEP 3 — Define seasons (adjust boundaries to match your calendar)
# ─────────────────────────────────────────────────────────────────────
season_bounds = {
    "Q1": (0,   90),
    "Q2": (90,  181),
    "Q3": (181, 272),
    "Q4": (272, n_days)
}

# ─────────────────────────────────────────────────────────────────────
# STEP 4 — K-means clustering within each season
# ─────────────────────────────────────────────────────────────────────
K           = 4
RANDOM_SEED = 42

results = []

for season, (start, end) in season_bounds.items():
    season_profiles = daily_profiles[start:end]
    km = KMeans(n_clusters=K, random_state=RANDOM_SEED, n_init=20)
    km.fit(season_profiles)

    labels    = km.labels_
    centroids = km.cluster_centers_

    # Sort clusters by mean load (ascending): d1=low, d4=high
    order = np.argsort(centroids.mean(axis=1))

    for rank, cluster_idx in enumerate(order):
        weight   = int(np.sum(labels == cluster_idx))
        centroid = centroids[cluster_idx]

        row = {"q": season, "d": f"d{rank + 1}", "weight": weight}
        for h in range(24):
            row[f"t{h + 1}"] = round(centroid[h], 1)
        results.append(row)

# ─────────────────────────────────────────────────────────────────────
# STEP 5 — Build output tables
# ─────────────────────────────────────────────────────────────────────
cols   = ["q", "d", "weight"] + [f"t{h}" for h in range(1, 25)]
df_out = pd.DataFrame(results, columns=cols)

# Weight table: same weight repeated across all 24 hour columns
df_weights = df_out[["q", "d"]].copy()
for h in range(1, 25):
    df_weights[f"t{h}"] = df_out["weight"]

# ─────────────────────────────────────────────────────────────────────
# STEP 6 — Print results
# ─────────────────────────────────────────────────────────────────────
print("\n=== Representative Day Profiles (centroid values) ===\n")
print(df_out.to_string(index=False))

print("\n=== Weight Table ===\n")
print(df_weights.to_string(index=False))

print("\n=== Verification: weight sums per season ===")
for season, (start, end) in season_bounds.items():
    s = df_out[df_out["q"] == season]["weight"].sum()
    expected = end - start
    status = "OK" if s == expected else "MISMATCH"
    print(f"  {season}: {s} / {expected} days  [{status}]")

# ─────────────────────────────────────────────────────────────────────
# STEP 7 — Export to CSV
# ─────────────────────────────────────────────────────────────────────
df_out.to_csv("representative_days_profiles.csv", index=False)
df_weights.to_csv("representative_days_weights.csv", index=False)
print("\nSaved: representative_days_profiles.csv")
print("Saved: representative_days_weights.csv")

#This document was generated by mAI.
